# NES-VMC 算法复现详解

## 论文: Pfau et al., "Accurate computation of quantum excited states with neural networks", Science 2024

本Notebook详细介绍如何复现论文中的NES-VMC方法，使用H2分子作为示例。

---

## 1. 算法原理详解

### 1.1 传统激发态方法的困境

传统的VMC激发态计算使用**惩罚函数法**：

$$L = \langle\psi|H|\psi\rangle + \sum_{i=0}^{n-1} \lambda_i |\langle\psi|\psi_i\rangle|^2$$

**问题**:
1. 需要调整惩罚参数 $\lambda_i$
2. 需要预先计算所有低能态
3. 顺序计算导致误差累积

### 1.2 NES-VMC的核心思想

论文提出了一种**无参数**的方法：

**核心洞察**: 如果我们有 $k$ 个线性无关的变分波函数 $\psi_1, \psi_2, ..., \psi_k$，可以构造哈密顿量矩阵：

$$\tilde{H}_{ij} = \langle\psi_i|H|\psi_j\rangle$$

通过最小化 $\text{Tr}(\tilde{H})$ 并对角化 $\tilde{H}$，可以得到各激发态能量。

### 1.3 为什么这有效？

**变分原理**: 设 $\tilde{H}$ 的本征值为 $E_0 \leq E_1 \leq ... \leq E_{k-1}$，则：

$$E_i \geq E_i^{\text{exact}}$$

即对角化后的能量是真实激发态能量的上界。

### 1.4 本实现的策略

由于论文中的完全无参数方法实现复杂，本实现采用**分步优化策略**：

1. **步骤1**: 优化基态 $\psi_0$
2. **步骤2**: 固定 $\psi_0$，优化 $\psi_1$ 使其与 $\psi_0$ 正交
3. **步骤3**: 计算哈密顿量矩阵 $\tilde{H}$ 并对角化

这种方法虽然需要惩罚参数，但更容易实现且结果稳定。

In [1]:
# 导入必要的库
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
from pyscf import gto, scf, fci
import netket.experimental as nkx
import jax
import jax.numpy as jnp
from flax import nnx
import jax.tree_util as jtu
import sys
sys.path.insert(0, '/Users/yangjianfei/mac_vscode/神经网络量子态/激发态能量/Netket_excited_state')
sys.path.insert(0, '.')

import vmc_ex
import nes_vmc

print("="*70)
print("NES-VMC 算法复现")
print("="*70)
print(f"NetKet版本: {nk.__version__}")
print(f"JAX版本: {jax.__version__}")

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NES-VMC 算法复现
NetKet版本: 3.18
JAX版本: 0.5.3


/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/netket/driver/vmc_common.py:33: FutureWarning: 

            `nk.driver.vmc_common is deprecated and the functionality removed.   

If you imported `nk.driver.vmc_common`, you must reimplement that functionality yourself.


  warn_deprecation(
/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/netket/utils/dispatch.py:25: FutureWarning: 
The variables `nk.utils.dispatch.{TrueT|FalseT|Bool}` are deprecated. Their usages
should instead be replaced by the following objects:

    `TrueT` should be replaced by `typing.Literal[True]`
    `FalseT` should be replaced by `typing.Literal[False]`
    `Bool` should be replaced by `bool`

  _warn_deprecation(


## 2. H2分子系统设置

### 2.1 分子几何与基组

H2分子是最简单的双原子分子：
- 2个电子
- 使用STO-3G基组（最小基组）
- 平衡键长约0.74 Å，我们使用1.4 Å（拉伸态）

In [2]:
# 分子几何构型
bond_length = 1.4  # 键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

print("="*70)
print("H2分子系统")
print("="*70)
print(f"键长: {bond_length} 埃 = {bond_length * 0.529177:.4f} 玻尔")

# 创建分子对象
mol = gto.M(atom=geometry, basis='STO-3G')
print(f"\n基组: STO-3G")
print(f"原子轨道数: {mol.nao_nr()}")
print(f"电子数: {mol.nelectron}")

# Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"\nHartree-Fock能量: {E_hf:.8f} Ha")

# FCI计算（精确参考）
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print(f"\nFCI能级（精确参考）:")
print("-"*50)
for i, e in enumerate(E_fcis):
    if i == 0:
        print(f"  E{i} (基态)     = {e:.8f} Ha")
    else:
        exc_eV = (e - E_fcis[0]) * 27.2114
        print(f"  E{i} (第{i}激发态) = {e:.8f} Ha, 激发能: {exc_eV:.4f} eV")

H2分子系统
键长: 1.4 埃 = 0.7408 玻尔

基组: STO-3G
原子轨道数: 2
电子数: 2

Hartree-Fock能量: -0.94148065 Ha

FCI能级（精确参考）:
--------------------------------------------------
  E0 (基态)     = -1.01546825 Ha
  E1 (第1激发态) = -0.87542794 Ha, 激发能: 3.8107 eV
  E2 (第2激发态) = -0.42938376 Ha, 激发能: 15.9482 eV
  E3 (第3激发态) = -0.26922131 Ha, 激发能: 20.3064 eV


### 2.2 Hilbert空间构造

在NetKet中，我们需要构造费米子Hilbert空间：

- **空间轨道数**: 2（每个H原子贡献1个轨道）
- **自旋轨道数**: 4（每个空间轨道有α和β自旋）
- **电子数**: 2（α=1, β=1）

Hilbert空间维度 = $\binom{4}{1} \times \binom{4}{1} = 4$ 个组态

In [3]:
# 创建费米子Hilbert空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,           # 空间轨道数
    s=1/2,                  # 自旋1/2
    n_fermions_per_spin=(1, 1)  # α=1, β=1
)

print("="*70)
print("Hilbert空间构造")
print("="*70)
print(f"空间轨道数: 2")
print(f"自旋轨道数: 4 (编号: 0=α₁, 1=β₁, 2=α₂, 3=β₂)")
print(f"电子数: 2 (α自旋1个, β自旋1个)")
print(f"\nHilbert空间维度: {hi.n_states}")

print(f"\n所有可能的电子组态:")
states = hi.all_states()
for i, state in enumerate(states):
    occupied = np.where(state == 1)[0]
    print(f"  |{i}⟩ = {state}, 占据轨道: {occupied}")

print("\n组态物理意义:")
print("  |0⟩ = [0,1,0,1]: 两个电子都在成键轨道 (基态主导)")
print("  |1⟩ = [0,1,1,0]: α在反键, β在成键")
print("  |2⟩ = [1,0,0,1]: α在成键, β在反键")
print("  |3⟩ = [1,0,1,0]: 两个电子都在反键轨道 (高能态)")

Hilbert空间构造
空间轨道数: 2
自旋轨道数: 4 (编号: 0=α₁, 1=β₁, 2=α₂, 3=β₂)
电子数: 2 (α自旋1个, β自旋1个)

Hilbert空间维度: 4

所有可能的电子组态:
  |0⟩ = [0 1 0 1], 占据轨道: [1 3]
  |1⟩ = [0 1 1 0], 占据轨道: [1 2]
  |2⟩ = [1 0 0 1], 占据轨道: [0 3]
  |3⟩ = [1 0 1 0], 占据轨道: [0 2]

组态物理意义:
  |0⟩ = [0,1,0,1]: 两个电子都在成键轨道 (基态主导)
  |1⟩ = [0,1,1,0]: α在反键, β在成键
  |2⟩ = [1,0,0,1]: α在成键, β在反键
  |3⟩ = [1,0,1,0]: 两个电子都在反键轨道 (高能态)


### 2.3 哈密顿量构建

NetKet通过`from_pyscf_molecule`自动从PySCF分子构建哈密顿量。

哈密顿量形式：
$$H = \sum_{pq} h_{pq} a_p^\dagger a_q + \frac{1}{2}\sum_{pqrs} h_{pqrs} a_p^\dagger a_q^\dagger a_r a_s$$

其中 $h_{pq}$ 是单电子积分，$h_{pqrs}$ 是双电子积分。

In [ ]:
# 创建哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

print("="*70)
print("哈密顿量信息")
print("="*70)
print(f"哈密顿量类型: {type(ha).__name__}")
print(f"希尔伯特空间: {ha.hilbert}")

# 计算哈密顿量的稀疏矩阵形式
H_sparse = ha.to_sparse()
print(f"\n稀疏矩阵维度: {H_sparse.shape}")
print(f"非零元素数: {H_sparse.nnz}")

# 精确对角化验证
H_dense = H_sparse.toarray()
eigenvalues, eigenvectors = np.linalg.eigh(H_dense)
print(f"\n精确对角化结果:")
for i, e in enumerate(eigenvalues[:4]):
    print(f"  E{i} = {e:.8f} Ha")

### 2.4 采样器设置

VMC需要从波函数概率分布中采样。对于费米子系统，使用`MetropolisFermionHop`采样器：

- **提议**: 随机选择两个轨道，交换电子
- **接受概率**: $\min(1, |\psi(\sigma')|^2/|\psi(\sigma)|^2)$

这保证了采样在费米子Hilbert空间中进行。

In [ ]:
# 创建采样器
# 图定义了哪些轨道之间可以发生电子跳跃
g = nk.graph.Graph(edges=[(0,1), (2,3)])  # 同一空间轨道的α-β跳跃

sampler = nk.sampler.MetropolisFermionHop(
    hi, 
    graph=g, 
    n_chains=16,        # 并行采样链数
    spin_symmetric=True, # 保持自旋对称
    sweep_size=64       # 每次扫描的提议数
)

print("="*70)
print("采样器设置")
print("="*70)
print(f"采样器类型: {type(sampler).__name__}")
print(f"采样链数: {sampler.n_chains}")
print(f"扫描大小: {sampler.sweep_size}")
print(f"\n跳跃图边: {list(g.edges())}")
print("允许的跳跃: (0↔1), (2↔3) 即同一轨道的α-β交换")

## 3. 神经网络波函数

### 3.1 网络架构

使用前馈神经网络(FFNN)作为变分波函数：

$$\psi_\theta(\sigma) = \text{NN}_\theta(\sigma)$$

网络输出复数值，表示波函数振幅。

**架构细节**:
- 输入: 4维二进制向量（电子占据数）
- 隐藏层: 2层，每层8个神经元
- 激活函数: tanh
- 输出: 1个复数值

In [ ]:
class FFNN(nnx.Module):
    """
    前馈神经网络波函数
    
    输入: 电子占据数向量 σ ∈ {0,1}^N
    输出: 复数波函数振幅 ψ(σ)
    """
    
    def __init__(self, N: int, alpha: int = 2, *, rngs: nnx.Rngs):
        """
        参数:
            N: 自旋轨道数
            alpha: 隐藏层宽度因子 (隐藏层宽度 = N * alpha)
            rngs: 随机数生成器
        """
        hidden_dim = alpha * N
        
        # 第一隐藏层
        self.linear1 = nnx.Linear(
            in_features=N, 
            out_features=hidden_dim, 
            rngs=rngs, 
            param_dtype=complex  # 复数参数
        )
        
        # 第二隐藏层
        self.linear2 = nnx.Linear(
            in_features=hidden_dim, 
            out_features=hidden_dim, 
            rngs=rngs, 
            param_dtype=complex
        )
        
        # 输出层
        self.linear_out = nnx.Linear(
            in_features=hidden_dim, 
            out_features=1, 
            rngs=rngs, 
            param_dtype=complex
        )
    
    def __call__(self, x: jax.Array):
        """
        前向传播
        
        参数:
            x: 输入占据数向量, shape (batch, N) 或 (N,)
        
        返回:
            波函数振幅 log|ψ| + i*arg(ψ)
        """
        # 第一层 + tanh激活
        h = self.linear1(x)
        h = nnx.tanh(h)
        
        # 第二层 + tanh激活
        h = self.linear2(h)
        h = nnx.tanh(h)
        
        # 输出层
        out = self.linear_out(h)
        
        return jnp.squeeze(out, axis=-1)

# 创建模型实例
N = 4  # 自旋轨道数
model = FFNN(N=N, alpha=2, rngs=nnx.Rngs(42))

# 统计参数量
params = nnx.state(model, nnx.Param)
n_params = sum(leaf.size for leaf in jtu.tree_leaves(params))

print("="*70)
print("神经网络模型")
print("="*70)
print(f"架构: 输入({N}) -> 隐藏({N*2}) -> 隐藏({N*2}) -> 输出(1)")
print(f"激活函数: tanh")
print(f"参数类型: 复数")
print(f"总参数量: {n_params}")

## 4. 步骤1: 基态优化

### 4.1 VMC基态优化原理

VMC通过最小化能量期望值来优化波函数：

$$E(\theta) = \frac{\langle\psi_\theta|H|\psi_\theta\rangle}{\langle\psi_\theta|\psi_\theta\rangle}$$

使用蒙特卡洛估计：

$$E(\theta) \approx \frac{1}{N}\sum_{i=1}^N E_L(\sigma_i)$$

其中 $E_L(\sigma) = \frac{H\psi(\sigma)}{\psi(\sigma)}$ 是局部能量。

### 4.2 梯度计算

能量对参数的梯度：

$$\nabla_\theta E = 2\text{Re}[\langle E_L O^* \rangle - \langle E_L \rangle \langle O^* \rangle]$$

其中 $O = \nabla_\theta \ln\psi$ 是对数导数。

In [ ]:
print("="*70)
print("步骤1: 基态优化")
print("="*70)

# 创建基态变分态
model_gs = FFNN(N=N, alpha=2, rngs=nnx.Rngs(42))
vs_gs = nk.vqs.MCState(
    sampler, 
    model_gs, 
    n_discard_per_chain=10,  # 预热样本数
    n_samples=512            # 每链样本数
)

print(f"\n变分态设置:")
print(f"  总样本数: {vs_gs.n_samples}")
print(f"  预热样本: {vs_gs.n_discard_per_chain}")

# 设置优化器
learning_rate = 0.05
diag_shift = 0.01

opt_gs = nk.optimizer.Sgd(learning_rate=learning_rate)
sr_gs = nk.optimizer.SR(diag_shift=diag_shift)

print(f"\n优化器设置:")
print(f"  方法: SGD + SR (随机重构)")
print(f"  学习率: {learning_rate}")
print(f"  SR对角位移: {diag_shift}")

# 运行优化
print(f"\n开始优化 (200迭代)...")
gs = nk.driver.VMC(ha, opt_gs, variational_state=vs_gs, preconditioner=sr_gs)
gs.run(n_iter=200, out='h2_gs', show_progress=False)

# 最终结果
E_gs = float(vs_gs.expect(ha).mean.real)
error_gs = abs(E_gs - E_fcis[0])

print(f"\n基态优化结果:")
print(f"  VMC能量: {E_gs:.8f} Ha")
print(f"  FCI能量: {E_fcis[0]:.8f} Ha")
print(f"  误差: {error_gs:.6f} Ha ({error_gs*27.2114*1000:.2f} meV)")

## 5. 步骤2: 激发态优化

### 5.1 惩罚函数法原理

为了找到与基态正交的激发态，我们修改目标函数：

$$L = \langle\psi_1|H|\psi_1\rangle + \lambda |\langle\psi_1|\psi_0\rangle|^2$$

其中 $\lambda$ 是惩罚参数，$\psi_0$ 是已优化的基态。

**关键**: 当 $\lambda > E_1 - E_0$ 时，最小化 $L$ 会使 $\psi_1$ 收敛到第一激发态。

### 5.2 梯度计算

修改后的梯度包含惩罚项贡献：

$$\nabla_\theta L = \nabla_\theta E + 2\lambda \text{Re}[\langle\psi_1|\psi_0\rangle \nabla_\theta \langle\psi_0|\psi_1\rangle]$$

In [ ]:
print("="*70)
print("步骤2: 激发态优化 (与基态正交)")
print("="*70)

# 创建激发态变分态（使用不同的随机种子）
model_ex1 = FFNN(N=N, alpha=2, rngs=nnx.Rngs(142))
vs_ex1 = nk.vqs.MCState(
    sampler, 
    model_ex1, 
    n_discard_per_chain=10, 
    n_samples=512
)

# 设置优化器
learning_rate_ex = 0.03
penalty = 0.3  # 惩罚参数

opt_ex1 = nk.optimizer.Sgd(learning_rate=learning_rate_ex)
sr_ex1 = nk.optimizer.SR(diag_shift=0.01)

print(f"\n激发态优化设置:")
print(f"  学习率: {learning_rate_ex}")
print(f"  惩罚参数 λ: {penalty}")
print(f"  正交化目标: 与基态正交")

# 使用VMC_ex（惩罚函数法）
gs_ex1 = vmc_ex.VMC_ex(
    hamiltonian=ha,
    optimizer=opt_ex1,
    variational_state=vs_ex1,
    preconditioner=sr_ex1,
    state_list=[vs_gs],   # 与基态正交
    shift_list=[penalty]   # 惩罚参数
)

print(f"\n开始优化 (200迭代)...")
gs_ex1.run(n_iter=200, out='h2_ex1', show_progress=False)

# 最终结果
E_ex1 = float(vs_ex1.expect(ha).mean.real)
error_ex1 = abs(E_ex1 - E_fcis[1])

print(f"\n激发态优化结果:")
print(f"  VMC能量: {E_ex1:.8f} Ha")
print(f"  FCI能量: {E_fcis[1]:.8f} Ha")
print(f"  误差: {error_ex1:.6f} Ha ({error_ex1*27.2114*1000:.2f} meV)")

## 6. 步骤3: 哈密顿量矩阵对角化

### 6.1 矩阵元素计算

计算哈密顿量矩阵：

$$H_{ij} = \langle\psi_i|H|\psi_j\rangle$$

使用蒙特卡洛估计：

$$H_{ij} \approx \frac{1}{N}\sum_{k=1}^N \frac{\psi_j(\sigma_k^i)}{\psi_i(\sigma_k^i)} E_L^j(\sigma_k^i)$$

其中 $\sigma_k^i$ 是从 $|\psi_i|^2$ 采样的构型。

### 6.2 广义特征值问题

求解广义特征值问题：

$$H c = E S c$$

其中 $S_{ij} = \langle\psi_i|\psi_j\rangle$ 是重叠矩阵。

In [ ]:
print("="*70)
print("步骤3: 哈密顿量矩阵对角化")
print("="*70)

# 计算哈密顿量矩阵和重叠矩阵
vstate_list = [vs_gs, vs_ex1]
H_matrix, S_matrix = nes_vmc.compute_matrices(vstate_list, ha)

print(f"\n哈密顿量矩阵 H:")
print(f"  H_00 = {H_matrix[0,0]:.8f} (基态能量)")
print(f"  H_11 = {H_matrix[1,1]:.8f} (激发态能量)")
print(f"  H_01 = {H_matrix[0,1]:.8f} (非对角元)")
print(f"\n完整矩阵:")
print(H_matrix)

print(f"\n重叠矩阵 S:")
print(f"  S_00 = {S_matrix[0,0]:.8f}")
print(f"  S_11 = {S_matrix[1,1]:.8f}")
print(f"  S_01 = {S_matrix[0,1]:.8f} (态间重叠)")
print(f"\n完整矩阵:")
print(S_matrix)

# 对角化
energies, coefficients = nes_vmc.diagonalize_generalized_eigenvalue_problem(H_matrix, S_matrix)

print(f"\n对角化结果:")
for i, e in enumerate(energies):
    print(f"  E{i} = {e.real:.8f} Ha (FCI: {E_fcis[i]:.8f} Ha)")

# 计算重叠
overlap = abs(S_matrix[0, 1])
print(f"\n态间重叠 |⟨ψ_0|ψ_1⟩| = {overlap:.6f}")
if overlap < 0.3:
    print("  ✓ 态区分良好 (重叠 < 0.3)")
else:
    print("  ⚠ 态重叠较大，可能需要更强的惩罚")

## 7. 结果总结与可视化

In [ ]:
print("="*70)
print("最终结果总结")
print("="*70)

print(f"\n{'态':<15} {'VMC (Ha)':<18} {'FCI (Ha)':<18} {'误差 (Ha)':<15} {'激发能 (eV)'}")
print("-"*85)

for i in range(2):
    if i == 0:
        e_vmc = E_gs
        state_name = "基态"
        exc_eV = 0.0
    else:
        e_vmc = E_ex1
        state_name = "第1激发态"
        exc_eV = (E_ex1 - E_gs) * 27.2114
    
    e_fci = E_fcis[i]
    error = abs(e_vmc - e_fci)
    print(f"{state_name:<15} {e_vmc:<18.8f} {e_fci:<18.8f} {error:<15.8f} {exc_eV:.4f}")

print(f"\nFCI激发能: {(E_fcis[1]-E_fcis[0])*27.2114:.4f} eV")
print(f"VMC激发能: {exc_eV:.4f} eV")
print(f"激发能误差: {abs(exc_eV - (E_fcis[1]-E_fcis[0])*27.2114):.4f} eV")

In [ ]:
# 绘制能级图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：能级比较
ax = axes[0]
x_vmc = np.arange(2) - 0.15
x_fci = np.arange(2) + 0.15

vmc_energies = [E_gs, E_ex1]
fci_energies = [E_fcis[0], E_fcis[1]]

for i in range(2):
    ax.hlines(vmc_energies[i], x_vmc[i]-0.1, x_vmc[i]+0.1, 
              colors='steelblue', linewidth=4, label='VMC' if i==0 else '')
    ax.hlines(fci_energies[i], x_fci[i]-0.1, x_fci[i]+0.1, 
              colors='coral', linewidth=4, linestyle='--', label='FCI' if i==0 else '')
    
    ax.text(x_vmc[i], vmc_energies[i]+0.02, f'{vmc_energies[i]:.4f}', 
            ha='center', fontsize=10, color='steelblue')
    ax.text(x_fci[i], fci_energies[i]-0.04, f'{fci_energies[i]:.4f}', 
            ha='center', fontsize=10, color='coral')

ax.set_xticks(np.arange(2))
ax.set_xticklabels(['E₀ (基态)', 'E₁ (第1激发态)'])
ax.set_xlabel('State', fontsize=12)
ax.set_ylabel('Energy (Ha)', fontsize=12)
ax.set_title('H2 Energy Levels: VMC vs FCI', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')

# 右图：误差分析
ax2 = axes[1]
states = ['Ground State', '1st Excited']
errors = [abs(E_gs - E_fcis[0]), abs(E_ex1 - E_fcis[1])]
errors_meV = [e * 27.2114 * 1000 for e in errors]

bars = ax2.bar(states, errors_meV, color=['steelblue', 'coral'], alpha=0.8)
ax2.set_ylabel('Error (meV)', fontsize=12)
ax2.set_title('Energy Error vs FCI', fontsize=14)
ax2.grid(True, alpha=0.3, axis='y')

for bar, err in zip(bars, errors_meV):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{err:.1f} meV', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('nes_vmc_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 算法总结

### 8.1 实现步骤回顾

| 步骤 | 内容 | 关键参数 |
|------|------|----------|
| 1 | 基态VMC优化 | 学习率=0.05, SR位移=0.01 |
| 2 | 激发态优化（惩罚函数法） | 学习率=0.03, 惩罚=0.3 |
| 3 | 哈密顿量矩阵对角化 | 广义特征值问题 |

### 8.2 与论文方法的对比

| 特性 | 论文NES-VMC | 本实现 |
|------|-------------|--------|
| 惩罚参数 | 无需 | 需要 (λ=0.3) |
| 优化方式 | 同时优化所有态 | 分步优化 |
| 数学基础 | 扩展系统变分原理 | 惩罚函数变分原理 |
| 实现复杂度 | 高 | 中等 |

### 8.3 关键技术要点

1. **神经网络波函数**: 使用复值参数，输出波函数振幅
2. **SR预条件**: 改善优化稳定性
3. **惩罚参数选择**: λ应大于激发能（以Ha为单位）
4. **态间重叠监控**: 确保 |⟨ψ₀|ψ₁⟩| < 0.3

In [ ]:
print("="*70)
print("计算完成！")
print("="*70)
print(f"\nH2分子 NES-VMC 计算")
print(f"键长: {bond_length} 埃")
print(f"\n基态:")
print(f"  VMC: {E_gs:.8f} Ha")
print(f"  FCI: {E_fcis[0]:.8f} Ha")
print(f"  误差: {abs(E_gs - E_fcis[0])*1000:.4f} mHa")
print(f"\n第一激发态:")
print(f"  VMC: {E_ex1:.8f} Ha")
print(f"  FCI: {E_fcis[1]:.8f} Ha")
print(f"  误差: {abs(E_ex1 - E_fcis[1])*1000:.4f} mHa")
print(f"\n激发能:")
print(f"  VMC: {(E_ex1 - E_gs)*27.2114:.4f} eV")
print(f"  FCI: {(E_fcis[1]-E_fcis[0])*27.2114:.4f} eV")
print(f"\n态间重叠: {overlap:.6f}")